In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd 

In [ ]:
### Process a L1b product from the PDS to before flat (ie undo radiometric cal, smooth shape, lab flat, image flat) 
### then compare with a product from my pipeline right before scattered light 

In [ ]:
# v03
# obs_id = 'm3g20090108t044645' 
# flat_id = "m3g20090108t044645"
# dark_id = "m3g20090108t040511"

# v02
obs_id = 'm3g20081201t064047' 
flat_id = "m3g20081201t064047"
dark_id = "m3g20081201t054112"

# pds l1b 
l1b_path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_l1b_rdn.fits"


In [ ]:
with fits.open(l1b_path) as hdul:
    image = hdul[0].data #.transpose(1, 0, 2)
    print(image.shape)

# will change depending on orientation of image 
image = np.flip(np.flip(image, axis=(2)).transpose(1, 0, 2), axis=(2))

obs_image = image[1000:1600,:,:].copy()
del image

In [ ]:
## check orientation of image re: flats etc 

fits.writeto(
                f"/home/bekah/m3-pipeline-dev/outputs/{obs_id}_check_orientation.fits", 
                obs_image, 
                overwrite=True
                )

l0_path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_l0.fits"

with fits.open(l0_path) as hdul:
    image = hdul[0].data.transpose(1, 0, 2)
    print(image.shape)

l0_image = image[1000:1600,:,:].copy()
del image

fits.writeto(
                f"/home/bekah/m3-pipeline-dev/outputs/{obs_id}_check_orientation_l0.fits", 
                obs_image, 
                overwrite=True
                )

In [ ]:
# undo smooth shape correction 

ssc_path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_ssc.txt"

ssc_table = pd.read_fwf(ssc_path, names=["channel", "corr_factor"])

obs_image = obs_image / ssc_table['corr_factor'].values[np.newaxis, :, np.newaxis]

# add rows and cols 

omitted_channels = [0]
left_col_cutoff = 9 
right_col_cutoff = 313

top_pad = np.max(omitted_channels) + 1
obs_image = np.pad(obs_image, 
                   ((0, 0), (top_pad, 0), (0, 0)), 
                   mode='constant', 
                   constant_values=0)


left_pad = left_col_cutoff
right_pad = 320 - right_col_cutoff 
obs_image = np.pad(obs_image, 
                   ((0, 0), (0, 0), (left_pad, right_pad)), 
                   mode='constant', 
                   constant_values=0)

# undo calibration coefficients 

rdn_cal_path = "/home/bekah/m3-pipeline-dev/l0_l1b_l2/cal_data/m3g20081118_rdn_cal.tab"

rdn_cal = pd.read_fwf(
        rdn_cal_path,
        names=['channel', 'rdn_cal_coeff'])

obs_image = obs_image / rdn_cal['rdn_cal_coeff'].values[np.newaxis, :, np.newaxis]

# undo image flat 

path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{flat_id}_ff.fits"

with fits.open(path) as hdul:
    image_flat = hdul[0].data
    
obs_image = obs_image * image_flat 

#undo lab flat 

path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/cal_data/lab_flat_field_global.fits"

with fits.open(path) as hdul:
    lab_flat = hdul[0].data
    
obs_image = obs_image / lab_flat 

fits.writeto(
                f"/home/bekah/m3-pipeline-dev/outputs/{obs_id}_reverse_l1b.fits", 
                obs_image, 
                overwrite=True
                )

In [ ]:
## run my version of the pipeline (I comment out things before SL in the code) 


DATA_ROOT = Path('/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data')  

l0_l1b_parent = Path("/home/bekah/m3-pipeline-dev")
if str(l0_l1b_parent) not in sys.path:
    sys.path.insert(0, str(l0_l1b_parent))

from l0_l1b_l2.pipeline import run_pipe


result_mission = run_pipe(
        obs_id=obs_id,
        pipe_version='mission',
        local_root=str(DATA_ROOT),
        save_steps=False,
        verbose=True
    )



In [ ]:
result_mission.shape

In [ ]:
fits.writeto(
                f"/home/bekah/m3-pipeline-dev/outputs/{obs_id}_mine.fits", 
                result_mission[1000:1600,:,:], 
                overwrite=True
                )

In [ ]:
fits.writeto(
                f"/home/bekah/m3-pipeline-dev/outputs/{obs_id}_div_2.fits", 
                result_mission[1000:1600,:,:]/obs_image, 
                overwrite=True
                )

In [ ]:
l0_path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{obs_id}_l0.fits"
dark_path = f"/home/bekah/m3-pipeline-dev/l0_l1b_l2/example_data/{dark_id}_l0.fits"

with fits.open(l0_path) as hdul:
    image = hdul[0].data.transpose(1, 0, 2)
    print(image.shape)

l0_image = image[1000:1600,:,:].copy()
del image

with fits.open(dark_path) as hdul:
    dark = np.mean(hdul[0].data.transpose(1, 0, 2)[10:, :, :], axis=0)

l0_image = l0_image - dark 

fits.writeto(
                f"/home/bekah/m3-pipeline-dev/outputs/{obs_id}_l0_minus_dark.fits", 
                l0_image, 
                overwrite=True
                )

In [ ]:
plt.figure(figsize=(10, 7))  

plt.plot(range(320), result_mission[1001, 43, :], ls=':', color='green')
plt.plot(range(320), result_mission[456, 23, :], ls=':', color='red')
plt.plot(range(320), result_mission[780, 10, :], ls=':', color='blue')
plt.plot(range(320), result_mission[600, 4, :], ls=':', color='orange')
plt.plot(range(320), result_mission[340, 32, :], ls=':', color='brown')

plt.axvline(9)
plt.axvline(312)

In [ ]:
plt.figure(figsize=(10, 7))  

col1 = 145 
col2 = 278

plt.plot(range(86), result_mission[1001, :, col1], ls=':', color='green')
plt.plot(range(86), result_mission[1001, :, col2], ls=':', color='blue')

plt.plot(range(86), obs_image[1, :, col1], color='green')
plt.plot(range(86), obs_image[1, :, col2], color='blue')

plt.plot(range(86), np.mean(l0_image[1, :, 314:316], axis=1), color='pink')
plt.plot(range(86), np.mean(l0_image[1, :, 6:8], axis=1), color='red')

plt.plot(range(86), np.mean(l0_image[1, :, 316:319], axis=1))
plt.plot(range(86), np.mean(l0_image[1, :, 2:5], axis=1))

#plt.plot(range(86), np.mean(l0_image[1, :, 10:316], axis=1), ls='--')

plt.plot(range(86), result_mission[1001, :, col1]-obs_image[1, :, col1], ls='-.', color='green')
plt.plot(range(86), result_mission[1001, :, col2]-obs_image[1, :, col2], ls='-.', color='blue')

plt.axvline(84)

In [ ]:
plt.plot(range(86), np.mean(l0_image[0, :, 314:316], axis=1)/np.mean(l0_image[0, :, 20:310], axis=1), ls='--')
plt.plot(range(86), np.mean(l0_image[0, :, 6:8], axis=1)/np.mean(l0_image[0, :, 20:310], axis=1), ls='--')

plt.plot(range(86), result_mission[1000, :, col1]/obs_image[0, :, col1], ls=':')
plt.plot(range(86), result_mission[1000, :, col2]/obs_image[0, :, col2], ls=':')

plt.axhline(1.0, color='grey')


In [ ]:
plt.plot(range(86), (np.mean(l0_image[0, :, 316:319], axis=1)/np.mean(l0_image[0, :, 10:316], axis=1))*result_mission[1000, :, col2], ls='--')
plt.axhline(0.0, color='red')
plt.axvline(84)

In [ ]:
plt.plot(range(86), ((np.mean(l0_image[0, :, 314:316], axis=1)/np.mean(l0_image[0, :, 10:300], axis=1))+1)*result_mission[1000, :, col2], ls='--')

plt.plot(range(86), result_mission[1000, :, col2], ls=':')
plt.plot(range(86), obs_image[0, :, col2])
